# ManimRAG Plugin Seed Corpus (v2)

End-to-end idempotent pipeline: **rich examples → ingest into RAG corpus**.

## Structure
```
seed_corpus/
  examples_v2/<plugin>.json          — CANONICAL rich examples (35 total, v2)
  examples/<plugin>/<api>/<nn>.json  — DEPRECATED thin one-liners (32, purged 2026-05-31)
  renders_v2/                        — render-verified videos for v2 examples
  render_and_verify.py               — render-gate runner
  seed.py                            — idempotent corpus ingestor (v2-aware)
  seed_corpus.ipynb                  — this notebook (primary seeding artifact)
  README.md                          — full usage docs
```

## Coverage (v2 — 35 rich multi-API examples)
| Plugin | Examples | Status |
|--------|----------|--------|
| manim-astronomy | 4 | ingested |
| manim-chemistry | 7 | ingested |
| manim-chess | 4 | ingested |
| manim-dsa | 7 | ingested |
| manim-ml | 7 | ingested |
| manim-physics | 6 | ingested |

**NOTE**: The thin `examples/` set (32 one-liners) was purged from the corpus on 2026-05-31.
Re-seeding this notebook always ingests from `examples_v2/` only.

## 0. Configuration

In [ ]:
import os
from pathlib import Path

# ─── Paths ────────────────────────────────────────────────────────────────────
SEED_DIR = Path(".").resolve()  # assumes notebook is in seed_corpus/
PIPELINE_DIR = SEED_DIR.parent
PLUGINS_ROOT = Path("/Users/vermarjun/Desktop/Manim plugins")

# ─── RAG service ──────────────────────────────────────────────────────────────
RAG_BASE_URL = os.environ.get("RAG_BASE_URL", "http://localhost:7001")
RAG_API_KEY  = os.environ.get("RAG_API_KEY", "change_me_rag_api_key")

print(f"SEED_DIR     : {SEED_DIR}")
print(f"PIPELINE_DIR : {PIPELINE_DIR}")
print(f"PLUGINS_ROOT : {PLUGINS_ROOT}")
print(f"RAG_BASE_URL : {RAG_BASE_URL}")

## 1. Verify RAG service is healthy

In [ ]:
import requests

resp = requests.get(f"{RAG_BASE_URL}/health", timeout=10)
health = resp.json()
print(f"Status       : {health['status']}")
vs = health.get('checks', {}).get('vectorStore', health.get('components', {}).get('vectorStore', {}))
print(f"VectorStore  : {vs.get('status','?')}  docs={vs.get('documentCount', '?')}")
assert health["status"] in ("ok", "degraded"), f"RAG service unhealthy: {health}"

## 2. Inspect existing examples

In [ ]:
import json
from collections import defaultdict

# ── v2 examples (canonical, RICH) ────────────────────────────────────────────
EXAMPLES_V2_DIR = SEED_DIR / "examples_v2"
PLUGINS = ["manim-astronomy", "manim-chemistry", "manim-chess", "manim-dsa", "manim-ml", "manim-physics"]

all_v2 = []
for plugin in PLUGINS:
    path = EXAMPLES_V2_DIR / f"{plugin}.json"
    if not path.exists():
        print(f"  MISSING: {path}")
        continue
    with open(path) as f:
        data = json.load(f)
    print(f"  {plugin}: {len(data)} examples  (ids: {[e['id'] for e in data[:2]]}...)")
    all_v2.extend(data)

print(f"\nTotal v2 examples: {len(all_v2)}")

# NOTE: examples/ (legacy thin set) is intentionally NOT loaded here.
# Those 32 one-liner docs were purged from the corpus on 2026-05-31.
print("\n[LEGACY examples/ dir is DEPRECATED — not loaded for ingestion]")

## 3. Render-gate: render any unverified examples

In [ ]:
import subprocess, sys

# v2 examples are pre-verified. render_and_verify.py is for the legacy thin set.
# For v2, renders are in renders_v2/ — check render_report.json for status.
RENDER_REPORT_V2 = SEED_DIR / "examples_v2" / "render_report.json"
if RENDER_REPORT_V2.exists():
    with open(RENDER_REPORT_V2) as f:
        report = json.load(f)
    print(f"v2 render report: {report}")
else:
    print("No render_report.json found in examples_v2/ — v2 examples are pre-verified by construction.")
    print("Skipping render-gate for v2 examples.")

## 4. Show render summary

In [ ]:
SUMMARY_PATH = SEED_DIR / "render_summary.json"

with open(SUMMARY_PATH) as f:
    summary = json.load(f)

totals = summary.get("totals", {})
results = summary.get("results", [])

print(f"Total: {totals.get('total',0)}  Pass: {totals.get('pass',0)}  Fail: {totals.get('fail',0)}  Skip: {totals.get('skip',0)}")
print()
for r in results:
    status = "SKIP" if r.get("skipped") else ("PASS" if r.get("verified") else "FAIL")
    print(f"  {status}  {r['plugin']:20s}/{r['api']:15s}/{r['nn']}")
    if status == "FAIL" and r.get("error"):
        print(f"         error: {r['error'].splitlines()[0][:80]}")

## 5. Dry-run ingestion (preview docs)

In [ ]:
# Dry-run: shows what would be sent (loads from examples_v2/, no HTTP calls)
result = subprocess.run(
    [sys.executable, str(SEED_DIR / "seed.py"),
     "--dry-run",
     "--rag-url", RAG_BASE_URL,
     "--rag-key", RAG_API_KEY],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])

## 6. LIVE ingestion — upsert all verified examples into RAG corpus

This cell is **idempotent**: re-running it will upsert (overwrite) existing docs
using stable IDs `plugin_seed_<plugin>_<api>_<nn>` — never creates duplicates.

In [ ]:
# LIVE ingestion — upsert all v2 examples into RAG corpus.
#
# Idempotency: the corpus server skips docs with existing IDs (skipReason=duplicate).
# Re-running this cell is always safe — it will STORE new docs and SKIP existing ones.
# Source: seed_corpus/examples_v2/<plugin>.json (35 rich examples across 6 plugins).
result = subprocess.run(
    [sys.executable, str(SEED_DIR / "seed.py"),
     "--rag-url", RAG_BASE_URL,
     "--rag-key", RAG_API_KEY],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[:1000])

## 7. Confirm corpus document count after ingestion

In [ ]:
resp = requests.get(f"{RAG_BASE_URL}/health", timeout=10)
health = resp.json()
vs = health.get('checks', {}).get('vectorStore', health.get('components', {}).get('vectorStore', {}))
doc_count = vs.get('documentCount', '?')
print(f"RAG service status : {health['status']}")
print(f"VectorStore docs   : {doc_count}")

# Also hit /corpus/stats
try:
    stats_resp = requests.get(
        f"{RAG_BASE_URL}/corpus/stats",
        headers={"x-api-key": RAG_API_KEY},
        timeout=10
    )
    stats = stats_resp.json()
    print(f"Corpus stats       : {stats}")
except Exception as e:
    print(f"(corpus/stats error: {e})")

## 8. Confirm queryable — probe /enhance with a seeded prompt

In [ ]:
# Use a prompt that closely matches one of our seeded docs
probe_prompt = "visualize a pendulum swinging using manim-physics SpaceScene"

resp = requests.post(
    f"{RAG_BASE_URL}/enhance",
    headers={"x-api-key": RAG_API_KEY, "Content-Type": "application/json"},
    json={
        "prompt": probe_prompt,
        "overrides": {
            "stages": {"queryTransform": False, "rerank": False, "pitfalls": False},
            "retrieval": {"denseTopK": 25}
        }
    },
    timeout=30
)

data = resp.json()
ctx = data.get("structuredContext", {})
api_examples = ctx.get("apiExamples", [])
similar_scripts = ctx.get("similarScripts", [])
trace = data.get("retrievalTrace", {})

print(f"Probe prompt        : {probe_prompt}")
print(f"Retrieval trace     : denseCandidates={trace.get('denseCandidates','?')} afterRerank={trace.get('afterRerank','?')}")
print(f"apiExamples returned: {len(api_examples)}")
print(f"similarScripts      : {len(similar_scripts)}")
print()
print("Top results:")
for ex in (api_examples + similar_scripts)[:5]:
    src = ex.get('source', ex.get('id','?'))
    sym = ex.get('symbol', ex.get('title', '?'))
    score = ex.get('score', 0)
    is_seed = 'plugin_seed' in src or 'plugin_seed' in str(ex.get('id',''))
    flag = " ← SEEDED" if is_seed else ""
    print(f"  source={src}  sym={str(sym)[:50]}  score={score:.3f}{flag}")

## 9. How to extend: add more examples for a new plugin

```python
# 1. Add examples to examples_v2/<plugin>.json
#    Required fields: id, originalPrompt, sceneClassName, title, description,
#    apisUsed, animationTypes, domain, category, tags, complexity, colorPalette,
#    quality (>=4), manimVersion, sourceType, plugin, code

# 2. Run a dry-run to preview the docs:
#    python seed.py --plugin <plugin> --dry-run

# 3. Ingest:
#    python seed.py --plugin <plugin>

# Or ingest everything at once:
#    python seed.py
```

**ID scheme**: stable `id` field in each v2 example (human-readable, e.g. `chem_plugin_bohr_atom_silicon`)
**Idempotency**: server skips docs with existing IDs (`skipReason=duplicate`) — re-running is always safe.
**Quality gate**: the server rejects docs with `quality < 4` (`skipReason=quality_too_low`).
**DEPRECATED**: `examples/<plugin>/<api>/<nn>.json` — thin one-liners purged 2026-05-31, do NOT re-ingest.